In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Sequential, layers, callbacks

In [2]:
df = pd.read_csv("../data/btc_1d_data_2018_to_2025.csv")

In [3]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
WINDOW = 30        
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

In [5]:
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
df = df.sort_values("Date").reset_index(drop=True)

In [8]:
df["ret_1d"] = np.log(df["Close"] / df["Close"].shift(1))      
df["hl_range"] = np.log(df["High"] / df["Low"])                
df["oc_change"] = np.log(df["Close"] / df["Open"])              
df["vol_change"] = np.log(df["Volume"] / df["Volume"].shift(1))

In [9]:
df["target"] = np.log(df["Close"].shift(-1) / df["Close"])


In [10]:
df = df.dropna().reset_index(drop=True)


In [11]:
FEATURES = ["ret_1d", "hl_range", "oc_change", "vol_change"]
TARGET = "target"

In [12]:
n = len(df)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

In [13]:
scaler = StandardScaler()
scaler.fit(train_df[FEATURES])

StandardScaler()

In [14]:
def scale(part):
    out = part.copy()
    out[FEATURES] = scaler.transform(part[FEATURES])
    return out

In [15]:
train_s, val_s, test_s = scale(train_df), scale(val_df), scale(test_df)

In [16]:
def make_windows(ctx):
    data = ctx[FEATURES].values
    targets = ctx[TARGET].values
    X, y, dates = [], [], []
    for i in range(WINDOW, len(ctx)):
        X.append(data[i - WINDOW:i])
        y.append(targets[i])
        dates.append(ctx["Date"].iloc[i])
    return np.array(X), np.array(y), np.array(dates)

In [17]:
X_train, y_train, _ = make_windows(train_s)
X_val, y_val, _ = make_windows(pd.concat([train_s.tail(WINDOW), val_s]).reset_index(drop=True))
X_test, y_test, test_dates = make_windows(pd.concat([val_s.tail(WINDOW), test_s]).reset_index(drop=True))

In [18]:
model = Sequential([
    layers.Input(shape=(WINDOW, len(FEATURES))),
    layers.SimpleRNN(32, activation="tanh"),
    layers.Dropout(0.2),
    layers.Dense(1),
])

In [19]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)               │ (None, 32)                  │           1,184 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,217 (4.75 KB)

 Trainable params: 1,217 (4.75 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
early = callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True
)

In [21]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early],
    shuffle=False,      
    verbose=0,
)

In [22]:
pred = model.predict(X_test, verbose=0).flatten()


In [24]:
def report(name, p, y):
    mae = np.mean(np.abs(p - y))
    rmse = np.sqrt(np.mean((p - y) ** 2))
    dir_acc = np.mean(np.sign(p) == np.sign(y))
    print(f"{name:<28} MAE={mae:.5f}  RMSE={rmse:.5f}  Yön_Dogr={dir_acc:.3f}")

In [25]:
y_test_raw = test_df[TARGET].values[-len(y_test):] 
assert np.allclose(y_test_raw, y_test), "Hizalama hatası: hedefler uyuşmuyor"

In [26]:
report("SimpleRNN", pred, y_test)
report("Baseline: sıfır getiri", np.zeros_like(y_test), y_test)

SimpleRNN                    MAE=0.02129  RMSE=0.02916  Yön_Dogr=0.492
Baseline: sıfır getiri       MAE=0.02044  RMSE=0.02834  Yön_Dogr=0.000


In [27]:
ret_1d_raw = test_df["ret_1d"].values[-len(y_test):]
report("Baseline: dünkü getiri", ret_1d_raw, y_test)
report("Baseline: ortalama getiri", np.full_like(y_test, y_train.mean()), y_test)

Baseline: dünkü getiri       MAE=0.03131  RMSE=0.04155  Yön_Dogr=0.470
Baseline: ortalama getiri    MAE=0.02045  RMSE=0.02832  Yön_Dogr=0.500


In [28]:
print(f"\nTest tahmin örnekleri (ilk 5):")
for d, p, t in list(zip(test_dates, pred, y_test))[:5]:
    print(f"  {pd.Timestamp(d).date()}  tahmin={p:+.4f}  gerçek={t:+.4f}")


Test tahmin örnekleri (ilk 5):
  2024-02-19  tahmin=+0.0035  gerçek=+0.0093
  2024-02-20  tahmin=-0.0009  gerçek=-0.0079
  2024-02-21  tahmin=-0.0055  gerçek=-0.0109
  2024-02-22  tahmin=-0.0030  gerçek=-0.0107
  2024-02-23  tahmin=+0.0003  gerçek=+0.0161


In [29]:
model.save("../model/bitcion_rnn.keras")